# Hierarchical Clustering - UPGMA

---
## Learning Objectives

1. Implement a distance-based phylogeny neighbor joining algorithm
1. Understanding of hierarchical clustering
1. Manipulation of arrays and lists in Python


---
## Imports
You will need to install a two new packages to properly render your phylogenetic trees. Using your preferred package manager, install both [`biopython`](https://biopython.org/) and `matplotlib` into your virtual environments.

For example:
```bash
$ micromamba create -n module06 -c bioconda -c conda-forge python numpy matplotlib biopython
```

In [ ]:
from typing import List, Tuple, Dict
from io import StringIO
import textdistance
import matplotlib.pyplot as plt
from Bio import Phylo
import numpy as np
from utils import cal_score, traceback 

---
## Background

Today we will implement a distance-based phylogenetic tree construction method using the Neighbor-Joining (NJ) algorithm and Smith-Waterman local alignment scores. Unlike UPGMA (Unweighted Pair Group Method using Arithmetic averages), NJ produces unrooted trees and does not assume a constant evolutionary rate across lineages, making it more biologically realistic for analyzing sequence relationships.

We will use HIV-1 reverse transcriptase sequences to construct our phylogenetic tree. The Smith-Waterman algorithm will be used to generate pairwise local alignment scores, which will then be converted into distances for tree construction. This approach is particularly suitable for HIV sequence analysis as it:
1. Handles sequence variations effectively through local alignment
2. Accounts for potential rate heterogeneity across different viral strains
3. Does not assume a molecular clock

The ultimate output will be an unrooted phylogenetic tree representing the evolutionary relationships between the HIV-1 sequences, visualized using the ete3 library. This method provides insights into viral diversity and evolutionary patterns while avoiding the assumptions of simpler hierarchical clustering approaches.

The key innovations of this implementation are:
- Use of Smith-Waterman for sensitive local alignment scoring
- Implementation of Neighbor-Joining for unrooted tree construction
- More biologically realistic evolutionary model

## Neighbor-Joining Algorithm

**Input**: Distance matrix $D$ for $n$ sequences

**Initialization**:
- Let $n$ be the number of sequences
- Assign each sequence $i$ to its own leaf node
- Initialize tree $T$ with leaf nodes

**Iteration**:
While $n > 2$:
1. Calculate Q-matrix where:
   $Q(i,j) = (n-2)d(i,j) - \sum_{k=1}^n d(i,k) - \sum_{k=1}^n d(j,k)$

2. Find pair $(i,j)$ with minimum $Q(i,j)$

3. Calculate branch lengths:
   $d_i = \frac{d(i,j) + (r_i - r_j)/(n-2)}{2}$
   $d_j = d(i,j) - d_i$
   where $r_i = \sum_{k=1}^n d(i,k)$

4. Create new node $k$
   - Add branches from $k$ to $i$ and $j$ with lengths $d_i$ and $d_j$

5. Update distances to remaining nodes $x$:
   $d(k,x) = \frac{d(i,x) + d(j,x) - d(i,j)}{2}$

6. Remove nodes $i$ and $j$
7. Add node $k$ to active nodes
8. $n = n - 1$

**Termination**:
When $n = 2$ with remaining nodes $i$ and $j$:
- Add final branch between $i$ and $j$ with length $d(i,j)$
- Return unrooted tree $T$

---
## Distance metrics for comparing sequences

### Smith-Waterman
This was previously implemented. Feel free to import if you know how...or Copy & Paste.



In [ ]:
def read_fasta(filename: str) -> Dict[str, str]:
    """Reads sequences from FASTA file.

    Args:
        filename (str): Path to FASTA file containing HIV RT sequences

    Returns:
        Dict[str, str]: Dictionary mapping sequence IDs to sequences

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> len(seqs) > 0
        True
    """
    seqs = {}
        # Open file
    with open(filename, 'r') as f:
        seq = ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):  # header line

                # Add sequence to dictionary
                if seq:
                    seqs[id] = seq
                    seq = ''

                # Extract id 
                id = line.split(' ')[-1]
            else:  # sequence line
                seq += line
                
        # Add last sequence
        seqs[id] = seq

    return seqs

In [ ]:
def smith_waterman(
    sequence1: str,
    sequence2: str,
    match_score: float = 2.0,
    mismatch_penalty: float = -1.0,
    gap_penalty: float = -1.0
) -> float:
    """Implements Smith-Waterman local alignment algorithm.

    Args:
        sequence1 (str): First HIV RT sequence
        sequence2 (str): Second HIV RT sequence
        match_score (float): Score for matching characters
        mismatch_penalty (float): Penalty for mismatched characters
        gap_penalty (float): Penalty for gaps

    Returns:
        float: Normalized similarity score between sequences

    Examples:
        >>> seq1 = "ACGT"
        >>> seq2 = "ACGT"
        >>> smith_waterman(seq1, seq2)
        1.0
    """
     # +1 on each side for the padding row and column of zeros
    n_rows = len(sequence1) + 1
    n_cols = len(sequence2) + 1

    # score_matrix holds the best local score ending at each cell
    score_matrix = np.zeros((n_rows, n_cols), dtype=int)

    # traceback_matrix holds the week 5 direction codes: 0 END, 1 DIAG, 2 UP, 3 LEFT
    # starts as all 0 (END), so the padding row/column automatically stops the traceback
    traceback_matrix = np.zeros((n_rows, n_cols), dtype=int)

    # Fill both matrices, starting at 1 to skip the padding
    # each cell only needs its up, left, and diagonal neighbors, which are already filled
    for i in range(1, n_rows):
        for j in range(1, n_cols):
            cell_score, direction = cal_score( score_matrix, sequence1, sequence2, i, j,
                match_score, mismatch_penalty, gap_penalty
            )
            score_matrix[i][j] = cell_score
            traceback_matrix[i][j] = direction

    # Local alignment ends at the highest scoring cell anywhere in the matrix (not the corner)
    # argmax gives a flat index, unravel_index turns it back into (row, col)
    best_cell_flat = np.argmax(score_matrix)
    best_cell = np.unravel_index(best_cell_flat, score_matrix.shape)

    # Walk back from the best cell until an END code to rebuild the aligned region
    aligned_seq1, aligned_seq2 = traceback(sequence1, sequence2, traceback_matrix, best_cell)

    # Fraction of aligned positions that differ (gaps count as differences since '-' never matches a residue)
    hamming_distance = textdistance.hamming.normalized_distance(aligned_seq1, aligned_seq2)

    return hamming_distance

In [ ]:
def build_distance_matrix(sequences: Dict[str, str]) -> Tuple[np.ndarray, List[str]]:
    """Builds DISTANCE (not similiarity) matrix using Smith-Waterman scores.

    Args:
        sequences (Dict[str, str]): Dictionary of HIV RT sequences

    Returns:
        Tuple[np.ndarray, List[str]]: Distance matrix and list of sequence IDs

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> mat[0:3, 0:3]  # Show 3x3 slice of distance matrix
        array([[0.000, 0.004, 0.012],
              [0.004, 0.000, 0.012],
              [0.012, 0.012, 0.000]])
        >>> len(ids)  # Number of sequences
        19
    """
    # Keep IDs and sequences in two lists with the same order,
    # so index k means the same sequence in the matrix and in seq_ids
    seq_ids = list(sequences.keys())
    seq_list = list(sequences.values())
    n_seqs = len(seq_ids)
 
    # Start with all zeros: the diagonal stays 0 because a sequence is 0 distance from itself
    distance_matrix = np.zeros((n_seqs, n_seqs))
 
    # Distance is symmetric (d(i,j) = d(j,i)), so only align each pair once:
    # col starts at row + 1, which skips the diagonal and the lower triangle
    for row in range(n_seqs):
        for col in range(row + 1, n_seqs):
 
            # Hamming distance of the local alignment
            pair_distance = smith_waterman(seq_list[row], seq_list[col])
 
            # Write the same value into both mirror cells (upper and lower triangle)
            distance_matrix[row][col] = pair_distance
            distance_matrix[col][row] = pair_distance
 
    return distance_matrix, seq_ids

In [ ]:
def build_q_matrix(distance_matrix: np.ndarray, n: int) -> Tuple[np.ndarray, np.ndarray]:
    """Builds the NJ Q-matrix and returns it with the row totals.
 
    Q(i,j) = (n-2) * d(i,j) - r_i - r_j, where r_i is the total distance from i to every node.
    """

    # r_i for every node
    row_totals = np.sum(distance_matrix, axis=1)
 
    q_matrix = np.zeros((n, n))
 
    # Q is symmetric like D, so I fill the upper triangle and mirror it
    # the diagonal stays 0
    for i in range(n):
        for j in range(i + 1, n):
            # subtracting r_i and r_j stops fast-evolving (long branch) sequences
            # from looking far from everything, which is how NJ avoids a molecular clock
            q_value = (n - 2) * distance_matrix[i][j] - row_totals[i] - row_totals[j]
            q_matrix[i][j] = q_value
            q_matrix[j][i] = q_value
 
    return q_matrix, row_totals

In [ ]:
def find_min_pair(q_matrix: np.ndarray, n: int) -> Tuple[int, int]:

    """Returns the indices (i, j) of the smallest Q value, upper triangle only."""
    
    # start at infinity so the first real Q value always replaces it
    min_q = float("inf")
    best_i, best_j = -1, -1
 
    for i in range(n):
        for j in range(i + 1, n):   # skip the diagonal and the mirrored lower triangle
            if q_matrix[i][j] < min_q:
                min_q = q_matrix[i][j]
                best_i, best_j = i, j
 
    return best_i, best_j

In [ ]:
def calculate_branch_lengths(distance_matrix: np.ndarray, row_totals: np.ndarray, i: int, j: int, n: int) -> Tuple[float, float]:
    """Limb lengths from the new internal node to i and to j."""

    d_ij = distance_matrix[i][j]
 
    # half of d(i,j), plus half of delta = (r_i - r_j) / (n - 2)
    
    branch_i = (d_ij + (row_totals[i] - row_totals[j]) / (n - 2)) / 2
    branch_j = d_ij - branch_i
 
    # negative lengths are set to 0 because they are not biologically meaningful
    branch_i = max(0.0, branch_i)
    branch_j = max(0.0, branch_j)
 
    return branch_i, branch_j

In [ ]:
def neighbor_joining(
    distance_matrix: np.ndarray,
    labels: List[str]
) -> str:
    """Implements Neighbor-Joining algorithm for phylogenetic tree construction.

    Args:
        distance_matrix (np.ndarray): Distance matrix from Smith-Waterman scores
        labels (List[str]): Sequence identifiers

    Returns:
        str: Newick format tree string

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> tree = neighbor_joining(mat, ids)
        >>> tree.startswith("((DM1:0.002,DM2:0.002)")  # Start of Newick string
        True
        >>> tree.count(",")  # Number of separators in tree
        18
    """
    # Work on copies so the caller's matrix and ID list do not changed
    current_matrix = distance_matrix.copy()
 
    # At the start every node is a leaf, so its text is just its ID.
    current_labels = list(labels)
 
    n = len(current_labels)
 
    while n > 2:
        # build Q and pick the pair to join
        q_matrix, row_totals = build_q_matrix(current_matrix, n)
        i, j = find_min_pair(q_matrix, n)
 
        #  branch lengths, computed BEFORE the matrix is updated
        branch_i, branch_j = calculate_branch_lengths(current_matrix, row_totals, i, j, n)
 
        # Step 4: the new node's label is the Newick text of the joined pair
        new_label = f"({current_labels[i]}:{branch_i:.3f},{current_labels[j]}:{branch_j:.3f})"
 
        # distance from the new node to every node we keep
        keep = [x for x in range(n) if x != i and x != j]
        new_distances = [
            (current_matrix[i][x] + current_matrix[j][x] - current_matrix[i][j]) / 2
            for x in keep
        ]
 
        # build the smaller matrix
        new_n = n - 1
        new_matrix = np.zeros((new_n, new_n))
        for new_row, old_row in enumerate(keep):
            for new_col, old_col in enumerate(keep):
                new_matrix[new_row][new_col] = current_matrix[old_row][old_col]
        for idx, dist in enumerate(new_distances):
            new_matrix[idx][new_n - 1] = dist
            new_matrix[new_n - 1][idx] = dist
 
        # Rebuild the labels in the SAME order as the matrix, so index k still means the same node
        current_labels = [current_labels[x] for x in keep] + [new_label]
        current_matrix = new_matrix
 
        n = new_n
 
    # Termination: 2 nodes left, joined by one edge of length d(0,1).
    # Newick needs a root to close the string, so the whole edge goes on the first node and 0 on the second. 
    # This root is only where the string is closed, NOT a biological root.
    final_length = max(0.0, current_matrix[0][1])
    newick_tree = f"({current_labels[0]}:{final_length:.3f},{current_labels[1]}:0.000);"
 
    return newick_tree

In [27]:
def plot_tree(newick_tree: str) -> None:
    """Plots a phylogenetic tree from a Newick string using Biopython and Matplotlib.

    The tree is rendered in a rectangular layout with smaller leaf labels and
    added margins to reduce overlap between labels and branches.

    Args:
        newick_tree (str): Tree in Newick format

    Returns:
        None: Displays the plotted tree
    """
    handle = StringIO(newick_tree)
    tree = Phylo.read(handle, "newick")

    fig, ax = plt.subplots(figsize=(8, 10))

    Phylo.draw(
        tree,
        axes=ax,
        do_show=False,
        label_func=lambda clade: clade.name if clade.is_terminal() else None,
    )
    for text in ax.texts:
        text.set_fontsize(6)              # shrink labels
    ax.margins(x=0.1, y=0.05)             # extra padding around tree

    plt.tight_layout()
    plt.show()


In [ ]:
if __name__ == "__main__":
    # Read HIV RT sequences
    sequences = read_fasta("lafayette_SARS_RT.fasta")
    
    # Build distance matrix using Smith-Waterman
    dist_matrix, seq_ids = build_distance_matrix(sequences)
    
    # Generate unrooted tree using Neighbor-Joining
    tree = neighbor_joining(dist_matrix, seq_ids)
    
    # Plot tree
    plot_tree(tree)